# 4. Evaluation and Analysis of Few-Shot Clustering

This notebook evaluates the clustering results and provides detailed analysis.

## Objectives
1. Load and analyze clustering results
2. Evaluate clustering performance metrics
3. Analyze cluster coherence and separation
4. Generate comprehensive reports
5. Compare with baseline (zero-shot) approach

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    silhouette_score, 
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)
from scipy.spatial.distance import cdist
from scipy.stats import entropy

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Libraries imported successfully")

## 1. Load Clustering Results

In [ ]:
# Load clustered data
df = pd.read_parquet('../output/grafana/clustered_logs.parquet')
embeddings = np.load('../output/grafana/embeddings.npy')

with open('../output/grafana/cluster_summary.json', 'r') as f:
    cluster_summary = json.load(f)

print(f"✅ Loaded clustering results")
print(f"\nDataset: {df.shape}")
print(f"Embeddings: {embeddings.shape}")
print(f"\nCluster Summary:")
print(json.dumps(cluster_summary, indent=2))

## 2. Comprehensive Clustering Metrics

In [ ]:
# Filter out noise points for metrics calculation
mask = df['cluster'] != -1
embeddings_clustered = embeddings[mask]
labels_clustered = df[mask]['cluster'].values

n_clusters = len(set(labels_clustered))

print("\n" + "="*80)
print("COMPREHENSIVE CLUSTERING EVALUATION")
print("="*80)

if n_clusters > 1:
    # Silhouette Score
    silhouette = silhouette_score(embeddings_clustered, labels_clustered)
    print(f"\n📊 Silhouette Score: {silhouette:.4f}")
    print(f"   Range: [-1, 1], Higher is better")
    print(f"   Interpretation: {'Excellent' if silhouette > 0.7 else 'Good' if silhouette > 0.5 else 'Fair' if silhouette > 0.25 else 'Poor'}")
    
    # Davies-Bouldin Index
    davies_bouldin = davies_bouldin_score(embeddings_clustered, labels_clustered)
    print(f"\n📊 Davies-Bouldin Index: {davies_bouldin:.4f}")
    print(f"   Range: [0, ∞], Lower is better")
    print(f"   Interpretation: {'Excellent' if davies_bouldin < 0.5 else 'Good' if davies_bouldin < 1.0 else 'Fair' if davies_bouldin < 2.0 else 'Poor'}")
    
    # Calinski-Harabasz Index
    calinski = calinski_harabasz_score(embeddings_clustered, labels_clustered)
    print(f"\n📊 Calinski-Harabasz Index: {calinski:.2f}")
    print(f"   Range: [0, ∞], Higher is better")
    print(f"   Interpretation: Measures cluster density and separation")
    
    # Cluster size balance (entropy)
    cluster_sizes = pd.Series(labels_clustered).value_counts().values
    cluster_entropy = entropy(cluster_sizes / cluster_sizes.sum())
    max_entropy = np.log(n_clusters)
    balance_score = cluster_entropy / max_entropy if max_entropy > 0 else 0
    
    print(f"\n📊 Cluster Balance Score: {balance_score:.4f}")
    print(f"   Range: [0, 1], Higher means more balanced cluster sizes")
    print(f"   Interpretation: {'Well-balanced' if balance_score > 0.8 else 'Moderately balanced' if balance_score > 0.6 else 'Imbalanced'}")
    
else:
    print("\n⚠️  Not enough clusters for comprehensive evaluation")

print("\n" + "="*80)

## 3. Per-Cluster Silhouette Analysis

In [ ]:
from sklearn.metrics import silhouette_samples

if n_clusters > 1:
    # Calculate silhouette score for each sample
    silhouette_values = silhouette_samples(embeddings_clustered, labels_clustered)
    
    # Create dataframe for analysis
    silhouette_df = pd.DataFrame({
        'cluster': labels_clustered,
        'silhouette': silhouette_values
    })
    
    # Per-cluster statistics
    cluster_silhouettes = silhouette_df.groupby('cluster')['silhouette'].agg(['mean', 'std', 'min', 'max'])
    
    print("\n📊 Per-Cluster Silhouette Scores:")
    print(cluster_silhouettes)
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Box plot
    silhouette_df.boxplot(column='silhouette', by='cluster', ax=axes[0])
    axes[0].set_title('Silhouette Score Distribution by Cluster', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Cluster', fontsize=12)
    axes[0].set_ylabel('Silhouette Score', fontsize=12)
    axes[0].axhline(y=0, color='r', linestyle='--', alpha=0.5, label='Zero line')
    plt.suptitle('')  # Remove auto title
    
    # Mean silhouette per cluster
    cluster_silhouettes['mean'].plot(kind='bar', ax=axes[1], color='steelblue')
    axes[1].set_title('Mean Silhouette Score by Cluster', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Cluster', fontsize=12)
    axes[1].set_ylabel('Mean Silhouette Score', fontsize=12)
    axes[1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
    axes[1].axhline(y=silhouette, color='g', linestyle='--', alpha=0.5, label='Overall mean')
    axes[1].legend()
    axes[1].tick_params(axis='x', rotation=0)
    
    plt.tight_layout()
    plt.savefig('../output/grafana/silhouette_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✅ Saved silhouette analysis to: output/grafana/silhouette_analysis.png")

## 4. Cluster Coherence Analysis

In [ ]:
def calculate_cluster_coherence(cluster_df):
    """
    Calculate coherence metrics for a cluster.
    """
    metrics = {}
    
    # Service diversity (entropy)
    service_counts = cluster_df['service'].value_counts()
    service_entropy = entropy(service_counts / service_counts.sum())
    metrics['service_entropy'] = service_entropy
    metrics['dominant_service_ratio'] = service_counts.iloc[0] / len(cluster_df) if len(service_counts) > 0 else 0
    
    # Dashboard diversity
    dashboard_counts = cluster_df['dashboard'].value_counts()
    dashboard_entropy = entropy(dashboard_counts / dashboard_counts.sum())
    metrics['dashboard_entropy'] = dashboard_entropy
    metrics['dominant_dashboard_ratio'] = dashboard_counts.iloc[0] / len(cluster_df) if len(dashboard_counts) > 0 else 0
    
    # Panel diversity
    panel_counts = cluster_df['panel_title'].value_counts()
    metrics['unique_panels'] = len(panel_counts)
    metrics['dominant_panel_ratio'] = panel_counts.iloc[0] / len(cluster_df) if len(panel_counts) > 0 else 0
    
    return metrics

# Calculate coherence for each cluster
coherence_data = []
for cluster_id in sorted(df[df['cluster'] != -1]['cluster'].unique()):
    cluster_df = df[df['cluster'] == cluster_id]
    coherence = calculate_cluster_coherence(cluster_df)
    coherence['cluster'] = cluster_id
    coherence['size'] = len(cluster_df)
    coherence_data.append(coherence)

coherence_df = pd.DataFrame(coherence_data)

print("\n📊 Cluster Coherence Metrics:")
print(coherence_df.to_string(index=False))

# Visualize coherence
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Service entropy
coherence_df.plot(x='cluster', y='service_entropy', kind='bar', ax=axes[0, 0], color='coral', legend=False)
axes[0, 0].set_title('Service Diversity (Entropy)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Cluster')
axes[0, 0].set_ylabel('Entropy')

# Dominant service ratio
coherence_df.plot(x='cluster', y='dominant_service_ratio', kind='bar', ax=axes[0, 1], color='steelblue', legend=False)
axes[0, 1].set_title('Dominant Service Ratio', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Cluster')
axes[0, 1].set_ylabel('Ratio')
axes[0, 1].set_ylim(0, 1)

# Dashboard entropy
coherence_df.plot(x='cluster', y='dashboard_entropy', kind='bar', ax=axes[1, 0], color='mediumseagreen', legend=False)
axes[1, 0].set_title('Dashboard Diversity (Entropy)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Cluster')
axes[1, 0].set_ylabel('Entropy')

# Unique panels
coherence_df.plot(x='cluster', y='unique_panels', kind='bar', ax=axes[1, 1], color='mediumpurple', legend=False)
axes[1, 1].set_title('Unique Panel Count', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Cluster')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../output/grafana/cluster_coherence.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Saved coherence analysis to: output/grafana/cluster_coherence.png")

## 5. Confusion Matrix: Few-Shot Labels vs Clusters

In [ ]:
# Create confusion matrix between few-shot labels and clusters
df_labeled = df[df['fewshot_label'].notna() & (df['cluster'] != -1)].copy()

if len(df_labeled) > 0:
    confusion_matrix = pd.crosstab(
        df_labeled['fewshot_label'],
        df_labeled['cluster'],
        margins=True
    )
    
    print("\n📊 Confusion Matrix: Few-Shot Labels vs Clusters")
    print(confusion_matrix)
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(
        confusion_matrix.iloc[:-1, :-1],  # Exclude margins
        annot=True,
        fmt='d',
        cmap='YlOrRd',
        ax=ax,
        cbar_kws={'label': 'Count'}
    )
    ax.set_title('Few-Shot Labels vs Cluster Assignment', fontsize=14, fontweight='bold')
    ax.set_xlabel('Cluster ID', fontsize=12)
    ax.set_ylabel('Few-Shot Label', fontsize=12)
    
    plt.tight_layout()
    plt.savefig('../output/grafana/confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✅ Saved confusion matrix to: output/grafana/confusion_matrix.png")

## 6. Cluster Size and Distribution Analysis

In [ ]:
# Cluster size analysis
cluster_sizes = df['cluster'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot
cluster_sizes[cluster_sizes.index != -1].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Cluster Size Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Cluster ID', fontsize=12)
axes[0].set_ylabel('Number of Logs', fontsize=12)
axes[0].tick_params(axis='x', rotation=0)

# Cumulative distribution
cluster_sizes_sorted = cluster_sizes[cluster_sizes.index != -1].sort_values(ascending=False)
cumulative = cluster_sizes_sorted.cumsum() / cluster_sizes_sorted.sum() * 100
axes[1].plot(range(1, len(cumulative) + 1), cumulative.values, marker='o', linewidth=2, color='coral')
axes[1].fill_between(range(1, len(cumulative) + 1), cumulative.values, alpha=0.3)
axes[1].set_title('Cumulative Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Top N Clusters', fontsize=12)
axes[1].set_ylabel('Cumulative % of Logs', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.savefig('../output/grafana/cluster_size_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Saved size distribution to: output/grafana/cluster_size_distribution.png")

# Statistics
sizes = cluster_sizes[cluster_sizes.index != -1].values
print(f"\n📊 Cluster Size Statistics:")
print(f"  Mean: {sizes.mean():.0f}")
print(f"  Median: {np.median(sizes):.0f}")
print(f"  Std: {sizes.std():.0f}")
print(f"  Min: {sizes.min()}")
print(f"  Max: {sizes.max()}")
print(f"  Top 3 clusters contain: {cumulative.iloc[2]:.1f}% of logs")

## 7. Generate Comprehensive Report

In [ ]:
# Generate comprehensive evaluation report
report = {
    'evaluation_timestamp': datetime.now().isoformat(),
    'dataset': {
        'total_logs': len(df),
        'clustered_logs': (df['cluster'] != -1).sum(),
        'noise_logs': (df['cluster'] == -1).sum(),
        'noise_percentage': float((df['cluster'] == -1).sum() / len(df) * 100)
    },
    'clustering_results': {
        'n_clusters': int(n_clusters),
        'cluster_size_mean': float(sizes.mean()),
        'cluster_size_std': float(sizes.std()),
        'cluster_size_min': int(sizes.min()),
        'cluster_size_max': int(sizes.max())
    },
    'quality_metrics': {
        'silhouette_score': float(silhouette) if 'silhouette' in locals() else None,
        'davies_bouldin_score': float(davies_bouldin) if 'davies_bouldin' in locals() else None,
        'calinski_harabasz_score': float(calinski) if 'calinski' in locals() else None,
        'cluster_balance_score': float(balance_score) if 'balance_score' in locals() else None
    },
    'few_shot_performance': {
        'labeled_logs': int(df['fewshot_label'].notna().sum()),
        'labeled_percentage': float(df['fewshot_label'].notna().sum() / len(df) * 100),
        'label_distribution': df['fewshot_label'].value_counts().to_dict()
    },
    'cluster_labels': cluster_summary.get('cluster_labels', {}),
    'recommendations': []
}

# Add recommendations based on metrics
if 'silhouette' in locals():
    if silhouette < 0.5:
        report['recommendations'].append("Consider adjusting HDBSCAN parameters (min_cluster_size, min_samples) for better cluster quality")
    if silhouette > 0.7:
        report['recommendations'].append("Excellent clustering quality - current parameters are working well")

if 'balance_score' in locals() and balance_score < 0.6:
    report['recommendations'].append("Clusters are imbalanced - consider using cluster_selection_method='leaf' in HDBSCAN")

if report['dataset']['noise_percentage'] > 20:
    report['recommendations'].append("High noise ratio - consider lowering min_cluster_size or using a different clustering algorithm")

# Save report
report_file = '../output/grafana/evaluation_report.json'
with open(report_file, 'w') as f:
    json.dump(report, f, indent=2)

print("\n" + "="*80)
print("COMPREHENSIVE EVALUATION REPORT")
print("="*80)
print(json.dumps(report, indent=2))
print("\n" + "="*80)

print(f"\n✅ Saved evaluation report to: {report_file}")

## 8. Summary and Conclusions

In [ ]:
print("\n" + "="*80)
print("SUMMARY AND CONCLUSIONS")
print("="*80)

print(f"\n✅ Successfully clustered {len(df):,} Grafana logs")
print(f"\n📊 Key Results:")
print(f"  - Identified {n_clusters} distinct log patterns")
print(f"  - Clustering quality: {report['quality_metrics'].get('silhouette_score', 'N/A')}")
print(f"  - Noise ratio: {report['dataset']['noise_percentage']:.1f}%")
print(f"  - Few-shot guidance: {report['few_shot_performance']['labeled_percentage']:.1f}% of logs matched examples")

print(f"\n🎯 Achievements:")
print(f"  - Automated log pattern discovery without manual labeling")
print(f"  - Semantic clustering based on metric names and queries")
print(f"  - Scalable approach using FAISS for similarity search")
print(f"  - Interpretable clusters with domain-guided labels")

if report['recommendations']:
    print(f"\n💡 Recommendations:")
    for i, rec in enumerate(report['recommendations'], 1):
        print(f"  {i}. {rec}")

print(f"\n📁 Generated Files:")
print(f"  - clustered_logs.parquet: Full dataset with cluster assignments")
print(f"  - embeddings.npy: Log embeddings for similarity search")
print(f"  - faiss_index.bin: FAISS index for fast retrieval")
print(f"  - cluster_summary.json: High-level clustering summary")
print(f"  - evaluation_report.json: Comprehensive evaluation metrics")
print(f"  - Multiple visualization PNGs")

print("\n" + "="*80)
print("✅ EVALUATION COMPLETE")
print("="*80)